# import, settings, file structure

In [ ]:
from copy import deepcopy
import os

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import fastplotlib as fpl

import mesmerize_core as mc
import mesmerize_viz

In [ ]:
# also need to install ipywidgets and ipympl
%matplotlib widget

# This is just a pandas table display formatting option
pd.options.display.max_colwidth = 120

# paths

In [ ]:
tsu_type = "org-" # "cab-" # 

In [ ]:
from pathlib import Path

main_path = Path("/Users/zzhao89/Documents/Git_Code/organoid/")
for _output_dir in ("mesmerize-batch", "mcorr"):
  (main_path / _output_dir).mkdir(parents=True, exist_ok=True)

mc.set_parent_raw_data_path(main_path)

batch_path = mc.get_parent_raw_data_path().joinpath(
  "mesmerize-batch/" + tsu_type + "mcorr" + ".pickle")


In [ ]:
df = mc.load_batch(batch_path)

# new
# df = mc.create_batch(batch_path)

# clear
# for idx, row in df.iterrows():
#   if row["algo"] == "mcorr":
#     df.caiman.remove_item(row.uuid)


df

In [ ]:
import tifffile

movie_path_s = [
  mc.get_parent_raw_data_path().joinpath(p.relative_to(main_path))
  for p in main_path.joinpath("tif/").iterdir() if p.name.startswith(tsu_type)]
movie_n = len(movie_path_s)

# mcorr

In [ ]:
mcorr_param_s = { 
  'main': { 
    'max_shifts': [50, 50], 
    'strides': [10, 10], 
    'overlaps': [5, 5],
    'max_deviation_rigid': 20,
    'border_nan': 'copy', 
    'pw_rigid': True, 
    'gSig_filt': None},}

# some variants of max_shifts
for movie_path in movie_path_s:
  # for strides in [40, 50, 30]:
  #   for shifts in [20, 30]:
  #     overlaps = int(strides / 2)
  #     # deep copy is the safest way to copy dicts
  #     new_param_s = deepcopy(mcorr_param_s)
  #     # assign the "max_shifts"
  #     new_param_s["main"]["max_shifts"] = (shifts, shifts)
  #     new_param_s["main"]["strides"] = (strides, strides)
  #     new_param_s["main"]["overlaps"] = (overlaps, overlaps)
  #     # add param variant to the batch
  #     df.caiman.add_item(
  #       algo='mcorr', 
  #       item_name=movie_path.stem, input_movie_path=movie_path,
  #       params=new_param_s)
  df.caiman.add_item(
    algo='mcorr', 
    item_name=movie_path.stem, input_movie_path=movie_path,
    params=mcorr_param_s)

In [ ]:
# when trying mulitiple parameters

# mcorr_param_diff = df.caiman.get_params_diffs(
#   algo="mcorr", item_name=movie_path.stem # df.iloc[0]["item_name"]
#   )
# mcorr_param_diff
# # df

In [ ]:
for idx, row in df.iterrows():
  if row["outputs"] is not None:
    continue # skip if item has already been run
  process = row.caiman.run()
  # on Windows you MUST reload the batch dataframe after every iteration because it uses the `local` backend.
  # this is unnecessary on Linux & Mac
  # "DummyProcess" is used for local backend so this is automatic
  if process.__class__.__name__ == "DummyProcess":
    df = df.caiman.reload_from_disk()

In [ ]:
# running updates on disk so reload
df = df.caiman.reload_from_disk()
df

## plot

In [ ]:
%matplotlib widget
# (Make sure: install ipywidgets ipympl)
import ipywidgets as widgets

# ---- Config ----
init_movie = 0               # initial iloc index
init_t = 0                   # initial frame index
cmap = "gray"                # change as you like

# ---- Helpers ----
def load_movie(i):
    # Replace this if your access path is different
    return df.iloc[int(i)].mcorr.get_output()

# Load initial movie
movie_vis = load_movie(init_movie)
t_max = len(movie_vis) - 1

# Make figure without auto-displaying a duplicate
plt.ioff()
fig, ax = plt.subplots()
img = ax.matshow(movie_vis[init_t], cmap=cmap)
ax.set_title(df.iloc[init_movie].item_name)
ax.set_xlabel("Time [frame]")
plt.ion()  # re-enable interactive mode for later updates

# ---- Update functions ----
def update_frame(t):
    t = int(t)
    # Guard against slider briefly exceeding current movie length during updates
    t = max(0, min(t, len(movie_vis)-1))
    img.set_data(movie_vis[t])

    ax.set_title(df.iloc[movie_slider.value].item_name)
    fig.canvas.draw_idle()

def on_movie_change(change):
    global movie_vis
    new_idx = int(change["new"])

    # Load new movie and reset the frame slider range/value
    movie_vis = load_movie(new_idx)
    new_t_max = len(movie_vis) - 1
    t_slider.max = max(0, new_t_max)
    # Keep current t if still valid, else snap to 0
    t_slider.value = min(t_slider.value, max(0, new_t_max))

    # Immediately refresh the image
    update_frame(t_slider.value)

def on_t_change(change):
    update_frame(change["new"])

# ---- Widgets ----
movie_slider = widgets.IntSlider(
    value=init_movie, min=0, max=movie_n-1, step=1,
    description="movie", continuous_update=True,
    layout=widgets.Layout(width="80%")
)
t_slider = widgets.IntSlider(
    value=init_t, min=0, max=t_max, step=1,
    description="time", continuous_update=True,
    layout=widgets.Layout(width="80%")
)

# Wire up callbacks
movie_slider.observe(on_movie_change, names="value")
t_slider.observe(on_t_change, names="value")

# Initialize once
update_frame(init_t)

# Layout
ui = widgets.VBox([
    widgets.HBox([movie_slider]),
    fig.canvas,
    widgets.HBox([t_slider]),
])
display(ui)

In [ ]:
mcorr_viz = df.mcorr.viz(
  data_options=["input", "mcorr"], 
  image_widget_kwargs={"grid_plot_kwargs": {"size": (1000, 500)}} 
  # you can also pass kwargs to the ImageWidget that is created
)

# mcorr_viz.show(sidecar=True)
mcorr_viz.show()

In [ ]:
mcorr_viz.close()

In [ ]:
# first item (first movie)
movie_s = [df.iloc[0].caiman.get_input_movie()]
movie_name_s = ["raw"]

# add all the mcorr outputs to the list
for idx, row in df.iterrows():
  # add movies to the list
  movie_s.append(row.mcorr.get_output())
  # movie_name_s.append(
  #   #f"idx{idx}..."
  #   "idx {0}: max_sh: {1}, str: {2}, ove: {3}".format(
  #     idx, 
  #     mcorr_param_diff.iloc[idx]["max_shifts"][0], 
  #     mcorr_param_diff.iloc[idx]["strides"][0], 
  #     mcorr_param_diff.iloc[idx]["overlaps"][0]))
  movie_name_s.append(
    f"idx{idx}")

# create the widget
mcorr_iw_multiple = fpl.ImageWidget(
  data=movie_s,  # list of movies
  # window functions as a kwarg, this is what the slider was used for in the ready-made viz
  window_funcs={"t": (np.mean, 17)}, 
  grid_plot_kwargs={"size": (900, 700)},
  names=movie_name_s,  # subplot names used for titles
  cmap="gnuplot2"
)

mcorr_iw_multiple.show()

In [ ]:
# for subplot in mcorr_iw_multiple.gridplot:
#     subplot.docks["right"].size = 0
# mcorr_iw_multiple.window_funcs["t"].window_size = 43

mcorr_iw_multiple.close()

# save

In [ ]:
plt.close("all")
fig, ax_s = plt.subplots(2, movie_n, figsize = (3*movie_n, 6))

import tifffile
for movie_idx in range(movie_n):
  # correct negative values after correction
  movie = df.iloc[movie_idx].mcorr.get_output()
  ax_s[0,movie_idx].hist(movie.ravel(), 30)
  movie = (np.maximum(0, movie) / np.max(movie) * 255).astype(np.uint8)
  ax_s[1,movie_idx].hist(movie.ravel(), 30)
  # save
  tifffile.imwrite(
    mc.get_parent_raw_data_path().joinpath("mcorr/" + df.iloc[movie_idx].item_name + ".tif"), 
    movie)